# Preprocessing Pipeline - Code Classification Challenge

**Objective**: Apply all preprocessing steps to prepare the dataset for modeling.

**Pipeline Steps**:
1. Text cleaning (patterns correction)
2. Translation (English normalization)
3. Near-duplicate detection and removal
4. Numeric variable conversion
5. Text/LaTeX separation
6. LaTeX feature extraction
7. Text concatenation (unified document)
8. Target encoding
9. Missing value imputation
10. Validation and summary


## 1. Setup and Data Loading


In [1]:
# Imports
import pandas as pd
import numpy as np
import warnings
from pathlib import Path
warnings.filterwarnings('ignore')

# Project utilities
import sys
sys.path.append('../')

from src.utils.eda_helpers import load_dataset, PRIORITY_TAGS
from src.utils.preprocessing import (
    clean_text_patterns,
    handle_difficulty_invalid_values,
    impute_missing_values,
    create_priority_tags_column,
    encode_multilabel_target,
    extract_latex_binary_features,
    create_text_length_features,
    remove_duplicate_rows,
    validate_preprocessing,
    print_preprocessing_step,
    get_preprocessing_summary
)
from src.utils.translation_helpers import translate_column
from src.utils.numeric_analysis import convert_time_limit_column
from src.utils.advanced_eda import detect_near_duplicates
from src.utils.text_analysis import preprocess_text_full, create_unified_document, remove_latex_from_text
from src.utils.latex_analysis import extract_all_latex_symbols

print("Setup complete")
print(f"Priority tags: {PRIORITY_TAGS}")


Setup complete
Priority tags: ['math', 'graphs', 'strings', 'number theory', 'trees', 'geometry', 'games', 'probabilities']


In [2]:
# Load raw dataset
DATA_DIR = '../data/raw/code_classification_dataset'
df = load_dataset(DATA_DIR)

n_initial = len(df)
print(f"\nInitial dataset: {n_initial:,} samples, {len(df.columns)} columns")
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")


Loaded 4982 samples from ../data/raw/code_classification_dataset

Initial dataset: 4,982 samples, 21 columns

Column names:
  - prob_desc_time_limit
  - prob_desc_sample_outputs
  - src_uid
  - prob_desc_notes
  - prob_desc_description
  - prob_desc_output_spec
  - prob_desc_input_spec
  - prob_desc_output_to
  - prob_desc_input_from
  - lang
  - lang_cluster
  - difficulty
  - file_name
  - code_uid
  - prob_desc_memory_limit
  - prob_desc_sample_inputs
  - exec_outcome
  - source_code
  - prob_desc_created_at
  - tags
  - hidden_unit_tests


## 2. Text Cleaning (Pattern Correction)

Correct common text formatting issues in the notes column.


In [3]:
# Clean prob_desc_notes patterns
n_before = len(df)
df = clean_text_patterns(df, column='prob_desc_notes')
n_after = len(df)

print_preprocessing_step("Text Pattern Cleaning", n_before, n_after)
print("\nPatterns corrected: NoteIN -> Note: In, NoteThe -> Note: The, etc.")



STEP: Text Pattern Cleaning
Samples before: 4,982
Samples after:  4,982

Patterns corrected: NoteIN -> Note: In, NoteThe -> Note: The, etc.


## 3. Translation (English Normalization)

Translate non-English text to English while preserving LaTeX formatting.


In [4]:
# Columns to translate
columns_to_translate = [
    'prob_desc_description',
    'prob_desc_input_spec',
    'prob_desc_output_spec',
    'prob_desc_notes'
]

print("="*100)
print("TRANSLATION TO ENGLISH")
print("="*100)

for col in columns_to_translate:
    print(f"\n{'-'*100}")
    print(f"Translating: {col}")
    print(f"{'-'*100}")
    
    df = translate_column(
        df,
        column=col,
        target_lang='en',
        new_column='_translated'
    )

print(f"\n{'='*100}")
print("ALL TRANSLATIONS COMPLETED")
print("="*100)
print("\nTranslated columns created:")
for col in columns_to_translate:
    print(f"  - {col}_translated")


TRANSLATION TO ENGLISH

----------------------------------------------------------------------------------------------------
Translating: prob_desc_description
----------------------------------------------------------------------------------------------------
🌐 Traduction de la colonne 'prob_desc_description' → 'prob_desc_description_translated'...
   Langue cible: en

✅ Traduction terminée:
   📝 Traduit:     18 / 4982 (  0.4%)
   ✓  Original:  4964 / 4982 ( 99.6%)
   ❌ Erreur:       0 / 4982 (  0.0%)

----------------------------------------------------------------------------------------------------
Translating: prob_desc_input_spec
----------------------------------------------------------------------------------------------------
🌐 Traduction de la colonne 'prob_desc_input_spec' → 'prob_desc_input_spec_translated'...
   Langue cible: en

✅ Traduction terminée:
   📝 Traduit:     29 / 4982 (  0.6%)
   ✓  Original:  4953 / 4982 ( 99.4%)
   ❌ Erreur:       0 / 4982 (  0.0%)

---------

## 4. Near-Duplicate Detection and Removal

Identify and remove near-duplicate problem descriptions to avoid data leakage.


In [5]:
# Detect near-duplicates on translated descriptions
print("\n" + "="*100)
print("NEAR-DUPLICATE DETECTION")
print("="*100)

dup_groups_df, df = detect_near_duplicates(
    df,
    column='prob_desc_description_translated',
    min_group_size=2
)

print(f"\nDuplicate groups found: {len(dup_groups_df)}")

if len(dup_groups_df) > 0:
    total_duplicates = dup_groups_df['count'].sum()
    print(f"Total samples in duplicate groups: {total_duplicates}")
    print(f"Samples to be removed: {total_duplicates - len(dup_groups_df)}")



NEAR-DUPLICATE DETECTION
🔍 Détection de near-duplicates sur 'prob_desc_description_translated'...

📊 Résultats:
   Textes normalisés uniques: 4973/4982 (99.8%)
   ⚠️  9 groupes de near-duplicates détectés
   📝 18 échantillons concernés (0.4%)

Duplicate groups found: 9
Total samples in duplicate groups: 18
Samples to be removed: 9


In [6]:
# Remove duplicates (keep one per group)
n_before = len(df)

# Get hash column name
hash_column = 'prob_desc_description_translated_hash'

if hash_column in df.columns:
    hash_counts = df[hash_column].value_counts()
    duplicate_hashes = hash_counts[hash_counts > 1].index
    
    non_duplicates = df[~df[hash_column].isin(duplicate_hashes)].copy()
    duplicates = df[df[hash_column].isin(duplicate_hashes)].copy()
    
    duplicates_kept = duplicates.drop_duplicates(subset=[hash_column], keep='first')
    
    df = pd.concat([non_duplicates, duplicates_kept], ignore_index=True)
    
    n_after = len(df)
    n_removed = n_before - n_after
    
    print_preprocessing_step("Near-Duplicate Removal", n_before, n_after)
    print(f"\nDuplicates removed: {n_removed}")
    print(f"Reduction rate: {n_removed/n_before*100:.2f}%")
else:
    print("\nNo duplicate detection performed (hash column not found)")



STEP: Near-Duplicate Removal
Samples before: 4,982
Samples after:  4,973
Change:         -9 (-0.18%)

Duplicates removed: 9
Reduction rate: 0.18%


## 4.1 Train/Test Split (Iterative Stratification)


In [20]:
# ============================================================================
# SECTION 4.1: TARGET ENCODING + TRAIN/TEST SPLIT (ITERATIVE STRATIFICATION)
# ============================================================================

print("\n" + "="*100)
print("SECTION 4.1: TARGET ENCODING & TRAIN/TEST SPLIT")
print("="*100)

# Step 1: Create priority tags column
print("\n[1/3] Creating priority tags column...")
df = create_priority_tags_column(
    df,
    tags_column='tags',
    priority_tags=PRIORITY_TAGS,
    new_column='tags_priority'
)

n_priority = df['tags_priority'].apply(len).gt(0).sum()
print(f"  Samples with priority tags: {n_priority} / {len(df)} ({n_priority/len(df)*100:.1f}%)")
print("  Note: Samples without priority tags are kept as negative examples")

# Step 2: Encode as binary multi-label
print("\n[2/3] Multi-label binary encoding...")
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
mlb.fit(df['tags_priority'])

encoded = mlb.transform(df['tags_priority'])
for i, tag in enumerate(mlb.classes_):
    df[f'target_{tag}'] = encoded[:, i]

target_cols = [col for col in df.columns if col.startswith('target_')]
print(f"  Binary target columns created: {len(target_cols)}")
print(f"  Classes: {list(mlb.classes_)}")

# Step 3: Train/Test Split with Iterative Stratification
print("\n[3/3] Train/Test Split with Iterative Stratification...")
from iterstrat.ml_stratifiers import MultilabelStratifiedShuffleSplit

# Extract targets for stratification
y_all = df[target_cols].values

# Configure split
msss = MultilabelStratifiedShuffleSplit(
    n_splits=1,
    test_size=0.2,  # 20% pour test
    random_state=42
)

# Perform stratified split
for train_idx, test_idx in msss.split(df, y_all):
    df_train = df.iloc[train_idx].copy()
    df_test = df.iloc[test_idx].copy()
    y_train = y_all[train_idx]
    y_test = y_all[test_idx]

# Display results
n_total = len(df)
print(f"\n✅ Split results:")
print(f"  Total:     {n_total:,} samples")
print(f"  Train set: {len(df_train):,} samples ({len(df_train)/n_total*100:.1f}%)")
print(f"  Test set:  {len(df_test):,} samples ({len(df_test)/n_total*100:.1f}%)")

# Verify stratification quality
print(f"\n📊 Label distribution verification:")
print(f"{'Label':<25} {'Train %':>10} {'Test %':>10} {'Diff':>10}")
print("-" * 60)

for i, col in enumerate(target_cols):
    tag_name = col.replace('target_', '').replace('_', ' ')
    train_pct = (y_train[:, i].sum() / len(y_train)) * 100
    test_pct = (y_test[:, i].sum() / len(y_test)) * 100
    diff = abs(train_pct - test_pct)
    print(f"{tag_name:<25} {train_pct:>9.2f}% {test_pct:>9.2f}% {diff:>9.2f}%")

# Average difference
avg_diff = np.mean([
    abs((y_train[:, i].sum() / len(y_train)) - (y_test[:, i].sum() / len(y_test))) 
    for i in range(y_all.shape[1])
]) * 100

print(f"\n📈 Stratification quality: {avg_diff:.3f}% average difference")
print("   (Lower = better stratification)")

# Label statistics
print(f"\n📊 Label statistics:")
print(f"  Labels/doc (train): {y_train.sum(axis=1).mean():.2f}")
print(f"  Labels/doc (test):  {y_test.sum(axis=1).mean():.2f}")

print("\n" + "="*100)
print("✅ TARGET ENCODING & SPLIT COMPLETED")
print("="*100)
print("\n💾 Objects to save for inference:")
print(f"  - mlb (MultiLabelBinarizer)")
print(f"  - target_cols: {target_cols}")
print("\n⚠️ From this point: All operations with fitting use TRAIN only!")
print("="*100)


SECTION 4.1: TARGET ENCODING & TRAIN/TEST SPLIT

[1/3] Creating priority tags column...
  Samples with priority tags: 2673 / 4973 (53.8%)
  Note: Samples without priority tags are kept as negative examples

[2/3] Multi-label binary encoding...
  Binary target columns created: 8
  Classes: ['games', 'geometry', 'graphs', 'math', 'number theory', 'probabilities', 'strings', 'trees']

[3/3] Train/Test Split with Iterative Stratification...

✅ Split results:
  Total:     4,973 samples
  Train set: 3,978 samples (80.0%)
  Test set:  995 samples (20.0%)

📊 Label distribution verification:
Label                        Train %     Test %       Diff
------------------------------------------------------------
games                          2.11%      2.11%      0.00%
geometry                       3.32%      3.32%      0.00%
graphs                        10.88%     10.85%      0.03%
math                          28.26%     28.24%      0.01%
number theory                  7.04%      7.04%      

## 5. Numeric Variable Conversion and Cleaning

Convert time_limit to float and handle invalid difficulty values.


In [21]:
# Convert time_limit to seconds (NO FITTING - applied to both)
print("\n" + "="*100)
print("TIME LIMIT CONVERSION (TRAIN + TEST)")
print("="*100)

print("\nProcessing TRAIN set...")
df_train = convert_time_limit_column(df_train, column='prob_desc_time_limit')

print("\nProcessing TEST set...")
df_test = convert_time_limit_column(df_test, column='prob_desc_time_limit')

print("\n Conversion complete for both train and test sets")



TIME LIMIT CONVERSION (TRAIN + TEST)

Processing TRAIN set...
🔄 Conversion de 'prob_desc_time_limit' en secondes...

✅ Conversion terminée:
   Converti : 3978 / 3978 (100.0%)
   Manquant :    0 / 3978 (0.0%)

📊 Valeurs uniques de time_limit_seconds:
      0.5s →   19 occurrences
      1.0s → 1548 occurrences
      1.5s →   45 occurrences
      2.0s → 1856 occurrences
      2.5s →   18 occurrences
      3.0s →  286 occurrences
      3.5s →    6 occurrences
      4.0s →  119 occurrences
      4.5s →    5 occurrences
      5.0s →   46 occurrences
      6.0s →   16 occurrences
      6.5s →    1 occurrences
      7.0s →    4 occurrences
      8.0s →    2 occurrences
      9.0s →    1 occurrences
     10.0s →    6 occurrences

Processing TEST set...
🔄 Conversion de 'prob_desc_time_limit' en secondes...

✅ Conversion terminée:
   Converti :  995 / 995 (100.0%)
   Manquant :    0 / 995 (0.0%)

📊 Valeurs uniques de time_limit_seconds:
      0.5s →    6 occurrences
      1.0s →  368 occurrences

In [22]:
# Handle invalid difficulty values (NO FITTING - applied to both)
print("\n" + "="*100)
print("DIFFICULTY CLEANING (TRAIN + TEST)")
print("="*100)

print("\nProcessing TRAIN set...")
n_invalid_train = (df_train['difficulty'] == -1).sum()
df_train = handle_difficulty_invalid_values(df_train, column='difficulty', invalid_value=-1)
n_nan_train = df_train['difficulty'].isna().sum()

print(f"  Invalid values (-1): {n_invalid_train}")
print(f"  Replaced with NaN: {n_nan_train}")

print("\nProcessing TEST set...")
n_invalid_test = (df_test['difficulty'] == -1).sum()
df_test = handle_difficulty_invalid_values(df_test, column='difficulty', invalid_value=-1)
n_nan_test = df_test['difficulty'].isna().sum()

print(f"  Invalid values (-1): {n_invalid_test}")
print(f"  Replaced with NaN: {n_nan_test}")

print("\nNote: Missing values will be imputed later (Section 11)")



DIFFICULTY CLEANING (TRAIN + TEST)

Processing TRAIN set...
  Invalid values (-1): 14
  Replaced with NaN: 46

Processing TEST set...
  Invalid values (-1): 1
  Replaced with NaN: 7

Note: Missing values will be imputed later (Section 11)


## 6. Text/LaTeX Separation

Extract clean text and LaTeX features from text columns.


In [23]:
# Apply text preprocessing on translated descriptions (NO FITTING - applied to both)
print("\n" + "="*100)
print("TEXT/LATEX SEPARATION (TRAIN + TEST)")
print("="*100)

print("\nProcessing TRAIN set: prob_desc_description_translated")
latex_analysis_train = df_train['prob_desc_description_translated'].apply(preprocess_text_full)

df_train['clean_description'] = latex_analysis_train.apply(lambda x: x[0])
df_train['latex_features_desc'] = latex_analysis_train.apply(lambda x: x[1])
df_train['nb_latex_blocks'] = df_train['latex_features_desc'].apply(lambda x: x['nb_latex_blocks'])
df_train['nb_latex_symbols'] = df_train['latex_features_desc'].apply(lambda x: x['nb_latex_symbols'])
df_train['latex_density'] = df_train['latex_features_desc'].apply(lambda x: x['latex_density'])
df_train['latex_symbols_density'] = df_train['latex_features_desc'].apply(lambda x: x['latex_symbols_density'])

n_latex_train = (df_train['nb_latex_blocks'] > 0).sum()
print(f"  Samples with LaTeX: {n_latex_train} / {len(df_train)} ({n_latex_train/len(df_train)*100:.1f}%)")

print("\nProcessing TEST set: prob_desc_description_translated")
latex_analysis_test = df_test['prob_desc_description_translated'].apply(preprocess_text_full)

df_test['clean_description'] = latex_analysis_test.apply(lambda x: x[0])
df_test['latex_features_desc'] = latex_analysis_test.apply(lambda x: x[1])
df_test['nb_latex_blocks'] = df_test['latex_features_desc'].apply(lambda x: x['nb_latex_blocks'])
df_test['nb_latex_symbols'] = df_test['latex_features_desc'].apply(lambda x: x['nb_latex_symbols'])
df_test['latex_density'] = df_test['latex_features_desc'].apply(lambda x: x['latex_density'])
df_test['latex_symbols_density'] = df_test['latex_features_desc'].apply(lambda x: x['latex_symbols_density'])

n_latex_test = (df_test['nb_latex_blocks'] > 0).sum()
print(f"  Samples with LaTeX: {n_latex_test} / {len(df_test)} ({n_latex_test/len(df_test)*100:.1f}%)")

print("\n Features created for both sets:")
print("  - clean_description, nb_latex_blocks, nb_latex_symbols")
print("  - latex_density, latex_symbols_density")



TEXT/LATEX SEPARATION (TRAIN + TEST)

Processing TRAIN set: prob_desc_description_translated
  Samples with LaTeX: 2217 / 3978 (55.7%)

Processing TEST set: prob_desc_description_translated
  Samples with LaTeX: 558 / 995 (56.1%)

 Features created for both sets:
  - clean_description, nb_latex_blocks, nb_latex_symbols
  - latex_density, latex_symbols_density


## 7. LaTeX Feature Extraction

Create binary features for the most common LaTeX symbols.


In [24]:
# Extract all LaTeX symbols (FIT ON TRAIN ONLY)
print("\n" + "="*100)
print("LATEX SYMBOL EXTRACTION (FIT ON TRAIN)")
print("="*100)

print("\n1. Extracting LaTeX symbols from TRAIN set...")
latex_stats_train = extract_all_latex_symbols(df_train, column='prob_desc_description_translated')

print(f"   LaTeX symbols found in TRAIN: {latex_stats_train.shape[1]} unique symbols")

print("\n2. Extracting LaTeX symbols from TEST set...")
latex_stats_test = extract_all_latex_symbols(df_test, column='prob_desc_description_translated')

print(f"   LaTeX symbols found in TEST: {latex_stats_test.shape[1]} unique symbols")

print("\n Note: Top N symbols will be selected from TRAIN data only")



LATEX SYMBOL EXTRACTION (FIT ON TRAIN)

1. Extracting LaTeX symbols from TRAIN set...
📝 112 symboles LaTeX uniques trouvés
   LaTeX symbols found in TRAIN: 112 unique symbols

2. Extracting LaTeX symbols from TEST set...
📝 64 symboles LaTeX uniques trouvés
   LaTeX symbols found in TEST: 64 unique symbols

 Note: Top N symbols will be selected from TRAIN data only


In [25]:
# Create binary features for top symbols (FIT ON TRAIN, APPLY TO BOTH)
print("\n" + "="*100)
print("LATEX BINARY FEATURES (FIT ON TRAIN, APPLY TO BOTH)")
print("="*100)

print("\n1. Selecting top symbols from TRAIN set...")
df_train = extract_latex_binary_features(
    df_train,
    latex_stats_train,
    top_n=30,
    min_frequency=10,
    prefix='has_'
)

has_features_train = [col for col in df_train.columns if col.startswith('has_')]
n_features_created = len(has_features_train)

print(f"   Binary features created: {n_features_created}")
print(f"   Top symbols selected from TRAIN data")

print("\n2. Identifying selected symbols from TRAIN...")
# Extract which symbols were selected (from column names)
# Column names are like 'has_le', 'has_ldots', etc.
# We need to map back to the original LaTeX symbol names in latex_stats_train
selected_symbols = []
for feat in has_features_train:
    # Remove 'has_' prefix to get the safe name
    safe_name = feat.replace('has_', '')
    # Find the original symbol in latex_stats_train columns
    # Symbols in latex_stats_train are like '\le', '\ldots', etc.
    for symbol in latex_stats_train.columns:
        symbol_safe = symbol.replace('\\', '').replace('{', '').replace('}', '')
        if symbol_safe == safe_name:
            selected_symbols.append(symbol)
            break

print(f"   Identified {len(selected_symbols)} symbols to apply to TEST")

print("\n3. Applying same symbols to TEST set...")
# Create features for TEST using the same symbols selected from TRAIN
for symbol in selected_symbols:
    safe_name = symbol.replace('\\', '').replace('{', '').replace('}', '')
    col_name = f"has_{safe_name}"
    
    # Check if symbol exists in latex_stats_test
    if symbol in latex_stats_test.columns:
        df_test[col_name] = latex_stats_test[symbol].values
    else:
        # Symbol not present in TEST, set all to 0
        df_test[col_name] = 0
        print(f"   ⚠️  Symbol {symbol} not found in TEST, set to 0")

has_features_test = [col for col in df_test.columns if col.startswith('has_')]

print(f"   Binary features created: {len(has_features_test)}")
print(f"   Same symbols as TRAIN applied to TEST")

# Verify alignment
missing_in_test = set(has_features_train) - set(has_features_test)
if missing_in_test:
    print(f"\n   ⚠️  WARNING: {len(missing_in_test)} features missing in TEST!")
    for feat in missing_in_test:
        print(f"      - {feat}")
else:
    print(f"\n   ✅ All {len(has_features_train)} features present in both TRAIN and TEST")

print(f"\n Feature names (examples from TRAIN):")
for feat in has_features_train[:10]:
    count_train = df_train[feat].sum()
    count_test = df_test[feat].sum() if feat in has_features_test else 0
    print(f"  - {feat}: {count_train} train ({count_train/len(df_train)*100:.1f}%), {count_test} test ({count_test/len(df_test)*100:.1f}%)")
if len(has_features_train) > 10:
    print(f"  ... and {len(has_features_train)-10} more")



LATEX BINARY FEATURES (FIT ON TRAIN, APPLY TO BOTH)

1. Selecting top symbols from TRAIN set...
   Binary features created: 30
   Top symbols selected from TRAIN data

2. Identifying selected symbols from TRAIN...
   Identified 30 symbols to apply to TEST

3. Applying same symbols to TEST set...
   Binary features created: 30
   Same symbols as TRAIN applied to TEST

   ✅ All 30 features present in both TRAIN and TEST

 Feature names (examples from TRAIN):
  - has_le: 732 train (18.4%), 177 test (17.8%)
  - has_ldots: 366 train (9.2%), 87 test (8.7%)
  - has_dots: 341 train (8.6%), 77 test (7.7%)
  - has_leq: 237 train (6.0%), 44 test (4.4%)
  - has_cdot: 200 train (5.0%), 52 test (5.2%)
  - has_ne: 146 train (3.7%), 30 test (3.0%)
  - has_times: 144 train (3.6%), 33 test (3.3%)
  - has_frac: 135 train (3.4%), 42 test (4.2%)
  - has_ge: 106 train (2.7%), 32 test (3.2%)
  - has_sum: 92 train (2.3%), 17 test (1.7%)
  ... and 20 more


## 8. Text Length Features

Create features based on text length characteristics.


In [26]:
from typing import List, Optional, Tuple, Dict
def create_text_length_features(df: pd.DataFrame,
                                 text_columns: List[str],
                                 compute_latex_ratio: bool = True,
                                 latex_density_columns: Optional[Dict[str, str]] = None) -> pd.DataFrame:
    """
    Create text length features for specified columns.
    
    Features created for each text column:
    - {col}_char_length: Total number of characters
    - {col}_word_count: Number of words (split by whitespace)
    - {col}_numeric_ratio: Proportion of numeric characters (0-9) in the text
    - {col}_latex_ratio: LaTeX density (if compute_latex_ratio=True)
    
    Parameters
    ----------
    df : pd.DataFrame
        Input dataframe
    text_columns : list of str
        Columns to compute length features for
    compute_latex_ratio : bool, default=True
        Whether to compute LaTeX ratio (requires latex_density columns)
    latex_density_columns : dict, optional
        Mapping {text_column: latex_density_column}
        
    Returns
    -------
    pd.DataFrame
        DataFrame with length features added
        
    Examples
    --------
    >>> df = create_text_length_features(df, ['prob_desc_description'])
    """
    df = df.copy()
    
    for col in text_columns:
        if col not in df.columns:
            continue
        
        df[f'{col}_char_length'] = df[col].apply(
            lambda x: len(str(x)) if pd.notna(x) else 0
        )
        
        df[f'{col}_word_count'] = df[col].apply(
            lambda x: len(str(x).split()) if pd.notna(x) else 0
        )
        
        df[f'{col}_numeric_ratio'] = df[col].apply(
            lambda x: sum(c.isdigit() for c in str(x)) / len(str(x)) if pd.notna(x) and len(str(x)) > 0 else 0.0
        )
        
        if compute_latex_ratio and latex_density_columns and col in latex_density_columns:
            latex_col = latex_density_columns[col]
            if latex_col in df.columns:
                df[f'{col}_latex_ratio'] = df[latex_col]
    
    return df


In [27]:
# Create text length features (NO FITTING - applied to both)
print("\n" + "="*100)
print("TEXT LENGTH FEATURES (TRAIN + TEST)")
print("="*100)

print("\nProcessing TRAIN set...")
df_train = create_text_length_features(
    df_train,
    text_columns=['prob_desc_description_translated'],
    compute_latex_ratio=True,
    latex_density_columns={'prob_desc_description_translated': 'latex_density'}
)

print("\nProcessing TEST set...")
df_test = create_text_length_features(
    df_test,
    text_columns=['prob_desc_description_translated'],
    compute_latex_ratio=True,
    latex_density_columns={'prob_desc_description_translated': 'latex_density'}
)

length_features = [
    'prob_desc_description_translated_char_length',
    'prob_desc_description_translated_word_count',
    'prob_desc_description_translated_latex_ratio',
    'prob_desc_description_translated_numeric_ratio'
]

print(f"\nFeatures created: {len(length_features)}")
print("\nFeature statistics:")
for feat in length_features:
    if feat in df_train.columns:
        mean_train = df_train[feat].mean()
        mean_test = df_test[feat].mean()
        print(f"  - {feat}")
        print(f"      Train mean: {mean_train:.2f}, Test mean: {mean_test:.2f}")
    else:
        print(f" {feat} not in columns")



TEXT LENGTH FEATURES (TRAIN + TEST)

Processing TRAIN set...

Processing TEST set...

Features created: 4

Feature statistics:
  - prob_desc_description_translated_char_length
      Train mean: 951.54, Test mean: 949.73
  - prob_desc_description_translated_word_count
      Train mean: 167.49, Test mean: 167.55
  - prob_desc_description_translated_latex_ratio
      Train mean: 0.10, Test mean: 0.09
  - prob_desc_description_translated_numeric_ratio
      Train mean: 0.01, Test mean: 0.01


## 9. Text Concatenation (Unified Document)

Create a unified document by concatenating all clean text columns.


## 9.1 Unified Document Without LaTeX

Create a version of the unified document with LaTeX replaced by a token, useful for models that don't handle LaTeX well.


In [28]:
# Create unified document from translated columns (NO FITTING - applied to both)
print("\n" + "="*100)
print("UNIFIED DOCUMENT CREATION (TRAIN + TEST)")
print("="*100)

print("\nProcessing TRAIN set...")
df_train['unified_document'] = df_train.apply(
    lambda row: create_unified_document(row, suffix="_translated"),
    axis=1
)
avg_length_train = df_train['unified_document'].apply(len).mean()
print(f"  Average length: {avg_length_train:.0f} characters")

print("\nProcessing TEST set...")
df_test['unified_document'] = df_test.apply(
    lambda row: create_unified_document(row, suffix="_translated"),
    axis=1
)
avg_length_test = df_test['unified_document'].apply(len).mean()
print(f"  Average length: {avg_length_test:.0f} characters")

print(f"\nExample from TRAIN (first 250 characters):")
print(df_train['unified_document'].iloc[0][:250] + "...")



UNIFIED DOCUMENT CREATION (TRAIN + TEST)

Processing TRAIN set...
  Average length: 2039 characters

Processing TEST set...
  Average length: 2011 characters

Example from TRAIN (first 250 characters):
[DESC] Numbers $$$1, 2, 3, \dots n$$$ (each integer from $$$1$$$ to $$$n$$$ once) are written on a board. In one operation you can erase any two numbers $$$a$$$ and $$$b$$$ from the board and write one integer $$$\frac{a + b}{2}$$$ rounded up instead...


In [29]:
# Create unified document WITHOUT LaTeX (NO FITTING - applied to both)
print("\n" + "="*100)
print("UNIFIED DOCUMENT WITHOUT LATEX (TRAIN + TEST)")
print("="*100)

print("\nProcessing TRAIN set...")
df_train['unified_document_without_latex'] = df_train['unified_document'].apply(
    lambda text: remove_latex_from_text(text, replacement_token='[LATEX]')
)
avg_with_train = df_train['unified_document'].apply(len).mean()
avg_without_train = df_train['unified_document_without_latex'].apply(len).mean()
reduction_train = (1 - avg_without_train / avg_with_train) * 100
print(f"  With LaTeX: {avg_with_train:.0f} chars, Without: {avg_without_train:.0f} chars")
print(f"  Reduction: {reduction_train:.1f}%")

print("\nProcessing TEST set...")
df_test['unified_document_without_latex'] = df_test['unified_document'].apply(
    lambda text: remove_latex_from_text(text, replacement_token='[LATEX]')
)
avg_with_test = df_test['unified_document'].apply(len).mean()
avg_without_test = df_test['unified_document_without_latex'].apply(len).mean()
reduction_test = (1 - avg_without_test / avg_with_test) * 100
print(f"  With LaTeX: {avg_with_test:.0f} chars, Without: {avg_without_test:.0f} chars")
print(f"  Reduction: {reduction_test:.1f}%")

n_latex_train = df_train['unified_document_without_latex'].str.contains('[LATEX]', regex=False).sum()
n_latex_test = df_test['unified_document_without_latex'].str.contains('[LATEX]', regex=False).sum()
print(f"\nDocuments with [LATEX] token:")
print(f"  Train: {n_latex_train} / {len(df_train)} ({n_latex_train/len(df_train)*100:.1f}%)")
print(f"  Test:  {n_latex_test} / {len(df_test)} ({n_latex_test/len(df_test)*100:.1f}%)")

print("="*100)


UNIFIED DOCUMENT WITHOUT LATEX (TRAIN + TEST)

Processing TRAIN set...
  With LaTeX: 2039 chars, Without: 1956 chars
  Reduction: 4.1%

Processing TEST set...
  With LaTeX: 2011 chars, Without: 1935 chars
  Reduction: 3.8%

Documents with [LATEX] token:
  Train: 3831 / 3978 (96.3%)
  Test:  954 / 995 (95.9%)


## 11. Missing Value Imputation

Impute missing values in numeric columns using median strategy.


In [30]:
# Impute missing values (FIT ON TRAIN, APPLY TO BOTH)
print("\n" + "="*100)
print("MISSING VALUE IMPUTATION (FIT ON TRAIN)")
print("="*100)

columns_to_impute = ['difficulty', 'time_limit_seconds']

print("\n1. Computing imputation values from TRAIN set...")
fill_values = {}
for col in columns_to_impute:
    if col in df_train.columns:
        fill_values[col] = df_train[col].median()

print("\nImputation values (median from TRAIN):") 
for col, value in fill_values.items():
    print(f"  - {col}: {value:.2f}")

print("\n2. Applying to TRAIN set...")
for col, value in fill_values.items():
    if col in df_train.columns:
        n_before = df_train[col].isna().sum()
        df_train[col].fillna(value, inplace=True)
        print(f"  - {col}: {n_before} missing values imputed")

print("\n3. Applying to TEST set (using TRAIN medians)...")
for col, value in fill_values.items():
    if col in df_test.columns:
        n_before = df_test[col].isna().sum()
        df_test[col].fillna(value, inplace=True)
        print(f"  - {col}: {n_before} missing values imputed")

print("\n IMPORTANT: Save fill_values for inference:")
print(f"   {fill_values}")
print("\nNote: These values were computed from TRAIN set only!")



MISSING VALUE IMPUTATION (FIT ON TRAIN)

1. Computing imputation values from TRAIN set...

Imputation values (median from TRAIN):
  - difficulty: 1700.00
  - time_limit_seconds: 2.00

2. Applying to TRAIN set...
  - difficulty: 46 missing values imputed
  - time_limit_seconds: 0 missing values imputed

3. Applying to TEST set (using TRAIN medians)...
  - difficulty: 7 missing values imputed
  - time_limit_seconds: 0 missing values imputed

 IMPORTANT: Save fill_values for inference:
   {'difficulty': np.float64(1700.0), 'time_limit_seconds': np.float64(2.0)}

Note: These values were computed from TRAIN set only!


## 13. Save Preprocessed Dataset

Save the preprocessed dataframe and imputation values for modeling.


In [31]:
# Display transformation summary
print("\n" + "="*100)
print("PREPROCESSING PIPELINE SUMMARY")
print("="*100)

print(f"\nInitial samples:     {n_initial:,}")
print(f"After deduplication: {n_initial - 9:,} (9 near-duplicates removed)")
print(f"  Train:             {len(df_train):,} ({len(df_train)/(n_initial-9)*100:.1f}%)")
print(f"  Test:              {len(df_test):,} ({len(df_test)/(n_initial-9)*100:.1f}%)")

print(f"\nFeatures created:")
print(f"  Initial:           21")
print(f"  Final (train):     {len(df_train.columns)}")
print(f"  Final (test):      {len(df_test.columns)}")
print(f"  Created:           {len(df_train.columns) - 21}")

# Verify train/test have same columns
if set(df_train.columns) == set(df_test.columns):
    print(f"\n  Column alignment: OK")
else:
    print(f"\n  WARNING: Train and test have different columns!")

print("\n" + "="*100)



PREPROCESSING PIPELINE SUMMARY

Initial samples:     4,982
After deduplication: 4,973 (9 near-duplicates removed)
  Train:             3,978 (80.0%)
  Test:              995 (20.0%)

Features created:
  Initial:           21
  Final (train):     78
  Final (test):      78
  Created:           57

  Column alignment: OK



In [32]:
# Save preprocessed datasets
print("\n" + "="*100)
print("SAVE PREPROCESSED DATASETS")
print("="*100)

output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

# Save train
train_path = output_dir / 'train_preprocessed.parquet'
df_train.to_parquet(train_path, index=False)
print(f"\nTrain set saved: {train_path}")
print(f"  Shape: {df_train.shape}")
print(f"  Columns: {len(df_train.columns)}")

# Save test
test_path = output_dir / 'test_preprocessed.parquet'
df_test.to_parquet(test_path, index=False)
print(f"\nTest set saved: {test_path}")
print(f"  Shape: {df_test.shape}")
print(f"  Columns: {len(df_test.columns)}")

# Save fill_values for inference
import json
fill_values_path = output_dir / 'imputation_values.json'
with open(fill_values_path, 'w') as f:
    json.dump(fill_values, f, indent=2)
print(f"\nImputation values saved: {fill_values_path}")
print(f"  Values: {fill_values}")

# Save MLB classes for inference
import pickle
mlb_path = output_dir / 'mlb_encoder.pkl'
with open(mlb_path, 'wb') as f:
    pickle.dump(mlb, f)
print(f"\nMultiLabel Binarizer saved: {mlb_path}")
print(f"  Classes: {list(mlb.classes_)}")

print("\n" + "="*100)
print("PREPROCESSING COMPLETE!")
print("="*100)
print(f"\nFinal datasets:")
print(f"  Train: {len(df_train):,} samples, {len(df_train.columns)} features")
print(f"  Test:  {len(df_test):,} samples, {len(df_test.columns)} features")
print(f"\nFiles saved in: {output_dir.absolute()}")
print("\nNext steps:")
print("  1. TF-IDF / Embeddings feature extraction")
print("  2. Model training and evaluation")
print("="*100)



SAVE PREPROCESSED DATASETS

Train set saved: ../data/processed/train_preprocessed.parquet
  Shape: (3978, 78)
  Columns: 78

Test set saved: ../data/processed/test_preprocessed.parquet
  Shape: (995, 78)
  Columns: 78

Imputation values saved: ../data/processed/imputation_values.json
  Values: {'difficulty': np.float64(1700.0), 'time_limit_seconds': np.float64(2.0)}

MultiLabel Binarizer saved: ../data/processed/mlb_encoder.pkl
  Classes: ['games', 'geometry', 'graphs', 'math', 'number theory', 'probabilities', 'strings', 'trees']

PREPROCESSING COMPLETE!

Final datasets:
  Train: 3,978 samples, 78 features
  Test:  995 samples, 78 features

Files saved in: /app/notebooks/../data/processed

Next steps:
  1. TF-IDF / Embeddings feature extraction
  2. Model training and evaluation
